# Ponderada 3

## Constru??o e deploy de um modelo de predi??o com dados de acidentes em rodovias de 2010 - 2023.

### Instala??o e importa??o de bibliotecas:


In [ ]:
!pip install pycaret[mlops]

In [ ]:
import pandas as pd
from google.colab import drive
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from pycaret.regression import *

### Extra??o dos dados


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
#drive.mount('/content/drive')
df = pd.read_csv("/content/Brazil Total highway crashes 2010 - 2023 - tratado.csv")
df

In [ ]:
df.columns

In [ ]:
df.describe

In [ ]:
df = df.fillna(0)
df.isna().sum()
df

In [ ]:
df = df.drop('tipo_de_ocorrencia', axis=1)

In [ ]:
df['tipo_de_acidente'].unique()

In [ ]:
contador = df['tipo_de_acidente'].str.upper().str.startswith('COLIS').sum()
print(contador)

In [ ]:
df.columns

In [ ]:
df['lugar_acidente'].unique()


In [ ]:
df['bicicleta'].unique()

In [ ]:
df['km'].unique()

In [ ]:
def faixa_km(km):
    if km >=0 and km <=200:
        return 1
    elif km >=201 and km <=400:
        return 2
    elif km >=401 and km <=600:
        return 3
    elif km >=601 and km <=800:
        return 4
    elif km >=801 and km <=1000:
        return 5
    else:
        return 6

df['faixa_km'] = df['km'].apply(faixa_km)

df['faixa_km']

In [ ]:
def tipo_de_acidente(acidente):
    if acidente == 0:
        return 0
    if acidente.upper().startswith('COLIS'):
        return 1
    elif acidente.upper().startswith('ATRO'):
        return 2
    elif acidente.upper().startswith('CHOQ'):
        return 3
    elif acidente.upper().startswith('ABAL'):
        return 4
    else:
        return 5

df['categoria_acidente'] = df['tipo_de_acidente'].apply(tipo_de_acidente)
df['categoria_acidente']

In [ ]:
df.columns

In [ ]:
df = df.drop('km', axis=1)

In [ ]:
df = df.drop('tipo_de_acidente', axis=1)

In [ ]:
df['data']=pd.to_datetime(df['data'], format='%d/%m/%Y')

def mes_ano(data):
    return data.strftime('%m/%Y')

df['mes_ano'] = df['data'].apply(lambda x: mes_ano(x))

df['mes_ano']

In [ ]:
df

In [ ]:
df.columns

In [ ]:
caminho_arquivo = 'Brazil: Total highway crashes 2010 - 2023/Brazil Total highway crashes 2010 - 2023 - tratado.csv'

In [ ]:
df.to_csv(caminho_arquivo, index=False)

In [ ]:
tipo = df['moto'][0]
print(type(tipo))

In [ ]:
df = df.drop(df.index[759156])

In [ ]:
df['categoria_acidente'] = pd.to_numeric(df['categoria_acidente'])

In [ ]:
df.columns

In [ ]:
features_to_normalize = ['automovel', 'bicicleta',
       'caminhao', 'moto', 'onibus', 'outros', 'tracao_animal',
       'transporte_de_cargas_especiais', 'trator_maquinas', 'utilitarios',
       'ilesos', 'levemente_feridos', 'moderadamente_feridos',
       'gravemente_feridos', 'mortos', 'faixa_km', 'categoria_acidente']


scaler = StandardScaler()

df[features_to_normalize] = scaler.fit_transform(df[features_to_normalize])
df

In [ ]:
amostra_aleatoria = df.sample(n=20000, random_state=42)

In [ ]:
s = setup(data = amostra_aleatoria, target = 'mortos')

In [ ]:
melhor_modelo = compare_models()

In [ ]:
melhor_modelo

In [ ]:
evaluate_model(melhor_modelo)

In [ ]:
save_model(melhor_modelo, 'modelo_ponderada3_teste')

In [ ]:
create_api(melhor_modelo, 'api_ponderada_teste')